In [1]:
import os
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage

load_dotenv(override=True)

groq_key = os.getenv("GROQ_APP_KEY")

In [ ]:
llm_compound = ChatGroq(model="groq/compound", api_key=groq_key, temperature=0)
llm_compound_mini = ChatGroq(model="groq/compound-mini", api_key=groq_key, temperature=0)

In [ ]:
system_prompt = """
You are an assistant that finds company contact emails for job applications.
I am looking to apply for a job/internship at this company.
Search the web for the most relevant emails. Prioritize in this order:
1. HR/recruitment emails (e.g. careers@, hr@, rrhh@)
2. Technical/data team emails
3. Generic emails (info@, contact@)
ONLY report emails you actually find through search.
NEVER invent or guess an email from general knowledge.
If you find an official career portal instead of a direct email, mention the URL.
If no email is found, state it clearly.
"""

test_companies = [
    {"company": "Cencosud", "city": "Santiago, Chile"},
    {"company": "Falabella", "city": "Santiago, Chile"},
    {"company": "Mercado Libre", "city": "Buenos Aires, Argentina"},
    {"company": "Axity", "city": "Región Metropolitana, Chile"},
]

In [ ]:
import time

def test_model(llm, model_name: str, companies: list[dict]):
    print(f"\n{'='*70}")
    print(f"MODEL: {model_name}")
    print('='*70)
    
    for c in companies:
        prompt = f"Find contact email(s) for {c['company']} located in {c['city']}, for a job application."
        
        t0 = time.time()
        try:
            response = llm.invoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=prompt)
            ])
            elapsed = time.time() - t0
            print(f"\n--- {c['company']} ({elapsed:.1f}s) ---")
            print(response.content)
        except Exception as e:
            print(f"\n--- {c['company']} ❌ ERREUR ---")
            print(str(e)[:300])

In [ ]:
test_model(llm_compound_mini, "groq/compound-mini", test_companies)

In [ ]:
test_model(llm_compound, "groq/compound", test_companies)

In [ ]:
print("\n\n" + "="*70)
print("COMPARAISON RAPIDE")
print("="*70)

for c in test_companies:
    prompt = f"Find contact email(s) for {c['company']} located in {c['city']}, for a job application."
    
    print(f"\n### {c['company']} ###")
    
    for llm, name in [(llm_compound_mini, "compound-mini"), (llm_compound, "compound")]:
        t0 = time.time()
        try:
            response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=prompt)])
            elapsed = time.time() - t0
            print(f"\n[{name}] ({elapsed:.1f}s):\n{response.content[:500]}")
        except Exception as e:
            print(f"\n[{name}] ❌ {str(e)[:200]}")